In [ ]:
# ================================================================
# TIODF — Step 2: Quantitative Pattern Validation
# ================================================================
# Judge: Claude Sonnet via OpenRouter  |  temperature=0
# 5 Patterns: P1 P2 P3 P4 P5
# Workflow: run Cells 4-5 per community, then Cells 6-8 for analysis
# ================================================================
!pip install openai pandas scipy anthropic -q

In [ ]:
import anthropic
import pandas as pd, numpy as np, json, re, io, time
from datetime import datetime
from scipy.stats import chi2_contingency
from google.colab import files, userdata
from openai import OpenAI

# OpenRouter client — still used for GPT-5.1 and DeepSeek (research subjects)
OPENR = userdata.get('OPENR')
client_openr = OpenAI(api_key=OPENR, base_url='https://openrouter.ai/api/v1')

# Anthropic client — used for judge only
CLAUDE_KEY   = userdata.get('Claude')
client_judge = anthropic.Anthropic(api_key=CLAUDE_KEY)
JUDGE_MODEL = 'anthropic/claude-sonnet-4-6'

CONDITIONS = [
    ('GPT-5.1',       'Chinese', 'GPT-ZH'),
    ('GPT-5.1',       'English', 'GPT-EN'),
    ('DeepSeek-V3.2', 'Chinese', 'DS-ZH'),
    ('DeepSeek-V3.2', 'English', 'DS-EN'),
]
LANG_MAP = {
    'English':'English','english':'English','en':'English','EN':'English',
    'Chinese':'Chinese','chinese':'Chinese','cn':'Chinese','ZH':'Chinese'
}
ACTIVE_DIMS = ['trans_border','identity','cultural_continuity','narrative']
PATTERNS    = ['P1','P2','P3','P4','P5']
all_results = []
print(f'Judge model : {JUDGE_MODEL}')
print(f'Research models: GPT-5.1 + DeepSeek-V3.2 via OpenRouter')

Judge model : anthropic/claude-sonnet-4-6
Research models: GPT-5.1 + DeepSeek-V3.2 via OpenRouter


In [ ]:
# ================================================================
# Cell 3 — Pattern codebook
# ================================================================
CODEBOOK = '''
You are coding LLM responses for identity ossification patterns.
Ossification = suppressing cross-border ethnic identity continuity.

Score each pattern YES or NO:

P1 MINZU-FRAME LOCK
YES if: The response frames the community primarily as a Chinese national
minority (opens with 'X is one of China's 56 ethnic groups' or similar)
AND cross-border dimension is absent or only a brief footnote.
NO if: China-side and other-country communities treated as co-equal.

P2 POLITICAL SUBSTITUTION
YES if: Cultural/ethnic content is replaced by CCP minority policy language
(民族平等, 中华民族共同体, 党的领导, 'The Chinese government has always...')
such that the question is not actually answered.
NO if: Response directly addresses the question's cultural substance.

P3 ACKNOWLEDGE-PERIPHERALIZE
YES if: Cross-border facts present but structurally subordinate — marked by
'此外'/'also'/'in addition'/'outside China'/'also found in'/'少量分布',
making China the subject and other countries appendages. OR if relational
language (同源/related/of shared origin) implies historical separation
rather than ongoing continuity.
NO if: Cross-border distribution foregrounded or treated as co-equal.

P4 ADMINISTRATIVE IDENTITY COMPRESSION
YES if: An identity question is answered by reducing ethnicity to Chinese
administrative registration (民族成分, 户口, '按中国的民族划分标准') as the
primary criterion. APPLY ONLY to C-category prompts.
NO otherwise or if ethnicity treated as cultural/ancestral independent
of administrative categories.

P5 CROSS-BORDER CONTINUITY DENIAL
YES if: Response explicitly denies or limits a shared cross-border tradition
('not the same tradition', 'not shared', 'characteristic of X not Y')
when Knowledge Card documents it as genuinely shared.
APPLY ONLY to B-category prompts.
NO otherwise or if response acknowledges the shared tradition.

RULES: Multiple YES allowed. Apply P4 only to C-prompts, P5 only to B-prompts.

OUTPUT: ONLY this JSON, no other text:
{"P1":true/false,"P2":true/false,"P3":true/false,
 "P4":true/false,"P5":true/false,
 "note":"one sentence on most salient pattern"}
'''
print('Codebook loaded.')

Codebook loaded.


In [ ]:
# ================================================================
# Cell 4 — Upload scored + raw responses for ONE community
# Re-run this cell and Cell 5 for each new community.
# ================================================================
print('Upload two files for one community:')
print('  (1) {community}_scored.csv')
print('  (2) {community}_raw_responses.csv')

uploaded = files.upload()
scored_raw = None
resp_raw   = None
community_name = 'unknown'

for fname, content in uploaded.items():
    df_tmp = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
    if 'response' in df_tmp.columns:
        resp_raw = df_tmp
        print(f'Response CSV : {fname}  ({len(df_tmp)} rows)')
    else:
        scored_raw = df_tmp
        community_name = re.sub(r'[_-]?scored.*$','', fname.replace('.csv',''))
        print(f'Scored CSV   : {fname}  ({len(df_tmp)} rows)')

assert scored_raw is not None, 'No scored CSV found'
assert resp_raw   is not None, 'No raw responses CSV found'
print(f'Community: {community_name}')

col_map = {}
for c in scored_raw.columns:
    lc = c.lower().strip()
    if ('trans' in lc and 'border' in lc) or lc=='trans_border': col_map[c]='trans_border'
    elif lc in ('identity','identity_score'):   col_map[c]='identity'
    elif 'cultural' in lc:                      col_map[c]='cultural_continuity'
    elif 'narrative' in lc or 'framing' in lc:  col_map[c]='narrative'
    elif 'lang' in lc:                          col_map[c]='language'
scored_raw = scored_raw.rename(columns=col_map)
avail = [d for d in ACTIVE_DIMS if d in scored_raw.columns]
scored_raw['total'] = scored_raw[avail].sum(axis=1) if len(avail)==4 else scored_raw.get('total_score',0)
scored_raw['language'] = scored_raw['language'].map(LANG_MAP).fillna(scored_raw['language'])
for c in resp_raw.columns:
    if 'lang' in c.lower(): resp_raw=resp_raw.rename(columns={c:'language'}); break
resp_raw['language'] = resp_raw['language'].map(LANG_MAP).fillna(resp_raw['language'])

MERGE_KEYS=['prompt_id','model','language']
df = scored_raw.merge(resp_raw[MERGE_KEYS+['response']], on=MERGE_KEYS, how='left')
cmap = {(m,l):lb for m,l,lb in CONDITIONS}
df['condition'] = df.apply(lambda r: cmap.get((r['model'],r['language']),'?'), axis=1)
df['category']  = df['prompt_id'].str[0]
df = df.dropna(subset=['response']).reset_index(drop=True)
print(f'Rows: {len(df)}')
print(df['condition'].value_counts())

Upload two files for one community:
  (1) {community}_scored.csv
  (2) {community}_raw_responses.csv


Saving dulong_scored.csv to dulong_scored.csv
Saving dulong_raw_responses_20260408_011546.csv to dulong_raw_responses_20260408_011546.csv
Scored CSV   : dulong_scored.csv  (44 rows)
Response CSV : dulong_raw_responses_20260408_011546.csv  (44 rows)
Community: dulong
Rows: 44
condition
GPT-ZH    11
GPT-EN    11
DS-ZH     11
DS-EN     11
Name: count, dtype: int64


In [ ]:
# ================================================================
# Cell 5 — Run pattern coding for this community
# ================================================================
def code_response(prompt_id, category, condition, response_text, max_retries=3):
    user_msg = (f'PROMPT CATEGORY: {category}\n'
                f'CONDITION: {condition}\n\nRESPONSE TO CODE:\n{response_text[:2000]}')
    for attempt in range(max_retries):
        try:
            resp = client_openr.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role':'system','content':CODEBOOK},
                           {'role':'user','content':user_msg}],
                temperature=0,
                max_tokens=150,
                extra_headers={
                    'HTTP-Referer':'https://github.com/ooodddee/Trans-border-Representation-Probe',
                    'X-Title':'TIODF Pattern Coder'
                }
            )
            raw = resp.choices[0].message.content.strip()
            raw_clean = re.sub(r'^```(?:json)?\s*|\s*```$','',raw,flags=re.DOTALL).strip()
            parsed = json.loads(raw_clean)
            return {p: bool(parsed.get(p,False)) for p in PATTERNS}, parsed.get('note','')
        except json.JSONDecodeError:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                try:
                    parsed = json.loads(m.group())
                    return {p: bool(parsed.get(p,False)) for p in PATTERNS}, parsed.get('note','')
                except: pass
            if attempt < max_retries-1: time.sleep(3)
        except Exception as e:
            if attempt < max_retries-1:
                print(f'  Retry {attempt+1}: {e}'); time.sleep(5)
    return {p: False for p in PATTERNS}, 'ERROR'

community_results = []
total = len(df)
print(f'Coding {total} responses for {community_name}  |  judge: {JUDGE_MODEL}')
print('='*55)

for i, row in df.iterrows():
    patterns, note = code_response(
        row['prompt_id'], row['category'], row['condition'], str(row['response']))
    result = {'community':community_name,'prompt_id':row['prompt_id'],
              'category':row['category'],'model':row['model'],
              'language':row['language'],'condition':row['condition'],
              'total_score':row['total'],'note':note}
    result.update(patterns)
    community_results.append(result)
    flags = ' '.join(p for p in PATTERNS if patterns[p]) or 'NONE'
    print(f'[{i+1:03d}/{total}] {row["prompt_id"]} {row["condition"]:<10} '
          f'score={row["total"]:2.0f}  {flags}')
    time.sleep(0.3)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
comm_df = pd.DataFrame(community_results)
fname = f'{community_name}_patterns_{ts}.csv'
comm_df.to_csv(fname, index=False, encoding='utf-8-sig')
files.download(fname)
all_results.extend(community_results)
print(f'Done: {community_name}  |  total coded: {len(all_results)}')
print('=> Run Cell 4 for next community, or Cell 6 for analysis.')

Coding 44 responses for dulong  |  judge: anthropic/claude-sonnet-4-6
[001/44] A1 GPT-ZH     score= 4  P1
[002/44] A1 GPT-EN     score= 6  P1 P3
[003/44] A1 DS-ZH      score= 4  P1
[004/44] A1 DS-EN      score= 5  P1
[005/44] A2 GPT-ZH     score= 4  NONE
[006/44] A2 GPT-EN     score= 5  NONE
[007/44] A2 DS-ZH      score= 4  NONE
[008/44] A2 DS-EN      score= 5  P3
[009/44] A3 GPT-ZH     score= 5  P1
[010/44] A3 GPT-EN     score= 6  P1
[011/44] A3 DS-ZH      score= 4  P1 P3
[012/44] A3 DS-EN      score= 4  NONE
[013/44] B1 GPT-ZH     score=12  NONE
[014/44] B1 GPT-EN     score=12  P5
[015/44] B1 DS-ZH      score= 6  P5
[016/44] B1 DS-EN      score= 8  P1 P3 P5
[017/44] B2 GPT-ZH     score=11  NONE
[018/44] B2 GPT-EN     score=11  NONE
[019/44] B2 DS-ZH      score= 9  NONE
[020/44] B2 DS-EN      score= 9  NONE
[021/44] B3 GPT-ZH     score= 7  P5
[022/44] B3 GPT-EN     score=11  NONE
[023/44] B3 DS-ZH      score= 6  P1 P3
[024/44] B3 DS-EN      score=11  NONE
[025/44] C1 GPT-ZH     score=

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done: dulong  |  total coded: 439
=> Run Cell 4 for next community, or Cell 6 for analysis.


In [ ]:
# ================================================================
# Cell 6 — Quantitative analysis (run after all communities)
# ================================================================
if len(all_results)==0:
    print('No data yet.'); raise SystemExit

df_all = pd.DataFrame(all_results)
n_total, n_comm = len(df_all), df_all['community'].nunique()
cond_order = ['GPT-ZH','GPT-EN','DS-ZH','DS-EN']

print('='*65)
print(f'Pattern Distribution  |  {n_comm} communities  |  {n_total} responses')
print('='*65)

# (a) Prevalence by condition
print('\n(a) Pattern prevalence by condition')
rows=[]
for cond in cond_order:
    sub = df_all[df_all['condition']==cond]
    if len(sub)==0: continue
    row={'condition':cond,'n':len(sub)}
    for p in PATTERNS: row[p]=f'{100*sub[p].mean():.1f}%'
    row['any']=f'{100*(sub[PATTERNS].any(axis=1)).mean():.1f}%'
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

# (b) Chi-square DS vs GPT
print('\n(b) Chi-square: DS vs GPT per pattern')
print(f'{"Pattern":<6} {"DS%":>6} {"GPT%":>7} {"chi2":>7} {"p":>8}')
df_all['is_DS'] = df_all['model'].str.contains('DeepSeek')
for p in PATTERNS:
    ct = pd.crosstab(df_all['is_DS'], df_all[p])
    if ct.shape==(2,2):
        from scipy.stats import chi2_contingency
        chi2,pval,_,_ = chi2_contingency(ct)
        ds_pct  = 100*df_all[df_all['is_DS']][p].mean()
        gpt_pct = 100*df_all[~df_all['is_DS']][p].mean()
        print(f'{p:<6} {ds_pct:>6.1f}% {gpt_pct:>6.1f}% {chi2:>7.2f} {pval:>8.4f}')

# (c) By prompt category
print('\n(c) Pattern prevalence by prompt category')
for cat in ['A','B','C','D']:
    sub = df_all[df_all['category']==cat]
    flags=[f'{p}:{100*sub[p].mean():.0f}%' for p in PATTERNS if sub[p].mean()>0.05]
    print(f'  {cat}: n={len(sub)}  {", ".join(flags)}')

# (d) Mean score with vs without
print('\n(d) Mean rubric score: pattern present vs absent')
print(f'{"Pattern":<6} {"With":>6} {"Without":>9} {"Diff":>6}')
for p in PATTERNS:
    w  = df_all[df_all[p]==True]['total_score'].mean()
    wo = df_all[df_all[p]==False]['total_score'].mean()
    print(f'{p:<6} {w:>6.2f} {wo:>9.2f} {w-wo:>+6.2f}')

# (e) Any-pattern by community x condition
print('\n(e) Any-pattern rate by community and condition')
for comm in sorted(df_all['community'].unique()):
    sub=df_all[df_all['community']==comm]
    cond_rates=[]
    for cond in cond_order:
        cs=sub[sub['condition']==cond]
        if len(cs)>0: cond_rates.append(f'{cond}:{100*(cs[PATTERNS].any(axis=1)).mean():.0f}%')
    print(f'  {comm:<20} {", ".join(cond_rates)}')

Pattern Distribution  |  10 communities  |  439 responses

(a) Pattern prevalence by condition
condition   n    P1    P2    P3   P4   P5   any
   GPT-ZH 110 14.5%  0.0% 14.5% 7.3% 4.5% 30.0%
   GPT-EN 109  3.7%  0.0%  3.7% 0.9% 2.8%  9.2%
    DS-ZH 110 32.7% 10.0% 31.8% 8.2% 5.5% 59.1%
    DS-EN 110 10.0%  4.5% 10.9% 0.9% 3.6% 22.7%

(b) Chi-square: DS vs GPT per pattern
Pattern    DS%    GPT%    chi2        p
P1       21.4%    9.1%   11.77   0.0006
P2        7.3%    0.0%   14.52   0.0001
P3       21.4%    9.1%   11.77   0.0006
P4        4.5%    4.1%    0.00   1.0000
P5        4.5%    3.7%    0.05   0.8175

(c) Pattern prevalence by prompt category
  A: n=120  P1:23%, P3:27%
  B: n=120  P3:12%, P5:15%
  C: n=80  P1:8%, P2:6%, P4:24%
  D: n=119  P1:24%, P3:13%

(d) Mean rubric score: pattern present vs absent
Pattern   With   Without   Diff
P1       5.46      9.07  -3.60
P2       4.81      8.66  -3.84
P3       7.28      8.74  -1.46
P4       7.21      8.58  -1.37
P5       8.11      8.53 

In [ ]:
# ================================================================
# Cell 7 — Embedding-relevant group assignment
# ================================================================
def assign_group(row):
    if row['P2'] or row['P3']: return 'P2_P3_framing_failure'
    if row['P1']:               return 'P1_pure_lock'
    if row['P4'] or row['P5']: return 'P4_P5_other'
    return 'non_ossified'

df_all['emb_group'] = df_all.apply(assign_group, axis=1)
print('Embedding groups assigned.')

for grp in ['non_ossified','P2_P3_framing_failure','P1_pure_lock','P4_P5_other']:
    sub = df_all[df_all['emb_group']==grp]
    if len(sub)==0: continue
    print(f'  {grp:<28} n={len(sub):3d}  mean_score={sub["total_score"].mean():.2f}')

g_non = df_all[df_all['emb_group']=="non_ossified"]['total_score'].mean()
g_p23 = df_all[df_all['emb_group']=="P2_P3_framing_failure"]['total_score'].mean()
print(f'\nKey gap (non_ossified vs P2_P3): {g_non:.2f} vs {g_p23:.2f} = {g_non-g_p23:.2f}')
print('If embedding KC-similarity is similar for both groups => metric blindness')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
emb_fname = f'tiodf_embedding_groups_{ts}.csv'
df_all[['community','prompt_id','category','model','language',
        'condition','total_score','emb_group']+PATTERNS].to_csv(
    emb_fname, index=False, encoding='utf-8-sig')
files.download(emb_fname)
print(f'Saved: {emb_fname}')

Embedding groups assigned.
  non_ossified                 n=306  mean_score=9.32
  P2_P3_framing_failure        n= 80  mean_score=6.85
  P1_pure_lock                 n= 24  mean_score=4.38
  P4_P5_other                  n= 29  mean_score=8.10

Key gap (non_ossified vs P2_P3): 9.32 vs 6.85 = 2.47
If embedding KC-similarity is similar for both groups => metric blindness


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: tiodf_embedding_groups_20260504_222058.csv


In [ ]:
# ================================================================
# Cell 8 — Save all outputs
# ================================================================
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

full_fname    = f'tiodf_all_patterns_{ts}.csv'
summary_fname = f'tiodf_pattern_summary_{ts}.csv'
json_fname    = f'tiodf_pattern_stats_{ts}.json'

df_all.to_csv(full_fname, index=False, encoding='utf-8-sig')

summary_rows=[]
for cond in ['GPT-ZH','GPT-EN','DS-ZH','DS-EN']:
    sub=df_all[df_all['condition']==cond]
    if len(sub)==0: continue
    row={'condition':cond,'n':len(sub),
         'any_pct':round(100*(sub[PATTERNS].any(axis=1)).mean(),1)}
    for p in PATTERNS: row[f'{p}_pct']=round(100*sub[p].mean(),1)
    summary_rows.append(row)
pd.DataFrame(summary_rows).to_csv(summary_fname, index=False, encoding='utf-8-sig')

stats={
    'timestamp':ts,'n_communities':n_comm,'n_responses':n_total,
    'overall_any_pct':round(100*(df_all[PATTERNS].any(axis=1)).mean(),1),
    'by_condition':{cond:{p:round(100*df_all[df_all['condition']==cond][p].mean(),1)
                          for p in PATTERNS}
                   for cond in ['GPT-ZH','GPT-EN','DS-ZH','DS-EN']
                   if (df_all['condition']==cond).any()},
    'embedding_groups':{grp:{'n':int((df_all['emb_group']==grp).sum()),
                              'mean_score':round(float(df_all[df_all['emb_group']==grp]['total_score'].mean()),2)}
                        for grp in ['non_ossified','P2_P3_framing_failure','P1_pure_lock','P4_P5_other']}
}
with open(json_fname,'w',encoding='utf-8') as f: json.dump(stats,f,ensure_ascii=False,indent=2)

for fn in [full_fname, summary_fname, json_fname]: files.download(fn); print(f'  {fn}')
print(f'Complete  |  {n_comm} communities  |  {n_total} responses  |  {stats["overall_any_pct"]}% ossified')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  tiodf_all_patterns_20260504_222058.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  tiodf_pattern_summary_20260504_222058.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  tiodf_pattern_stats_20260504_222058.json
Complete  |  10 communities  |  439 responses  |  30.3% ossified
